In [1]:
import pandas as pd
import seaborn as sns
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import plotly.offline as pyo
import plotly.graph_objs as go
import plotly.express as px
from sklearn.preprocessing import RobustScaler, StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from scipy.stats.mstats import winsorize
from sklearn.feature_selection import VarianceThreshold 

In [7]:
train = pd.read_csv('train_optimal.csv')
test = pd.read_csv('test_optimal.csv')
train_og = pd.read_csv('train.csv')
test_og = pd.read_csv('test.csv')

In [3]:
train.head()

,Emisi Savanna Api,Emisi Tanah Organik Yang Dikeringkan (Co2),Emisi Pembuatan Pestisida,Emisi Transportasi Makanan,Lahan Hutan,Konversi Hutan Bersih,Emisi Ritel Makanan,Emisi Pupuk Kandang Di Padang Rumput,Emisi Kebakaran Di Tanah Organik,Emisi Total,Peningkatan Suhu Rata - Rata ° C,Tahun,Negara_Encoded
0,2.816827,0.693147,2.625211,4.175130,-13.367805,0.0,4.715320,7.373080,0.0,7.902014,0.536167,1990,0
1,2.816827,0.693147,2.618277,4.145443,-13.367805,0.0,4.776421,7.414112,0.0,7.947195,0.020667,1991,0
2,2.816827,0.693147,2.618277,4.011870,-13.367805,0.0,4.853373,7.411862,0.0,7.958598,-0.259583,1992,0
3,2.816827,0.693147,2.618277,4.030602,-13.367805,0.0,4.424375,7.405472,0.0,7.962843,0.101917,1993,0
4,2.816827,0.693147,2.618277,4.023931,-13.367805,0.0,4.526135,7.433287,0.0,8.007875,0.372250,1994,0


In [4]:
test.head()

,Emisi Savanna Api,Emisi Tanah Organik Yang Dikeringkan (Co2),Emisi Pembuatan Pestisida,Emisi Transportasi Makanan,Lahan Hutan,Konversi Hutan Bersih,Emisi Ritel Makanan,Emisi Pupuk Kandang Di Padang Rumput,Emisi Kebakaran Di Tanah Organik,Emisi Total,Tahun,Negara_Encoded
0,1.045704,0.693147,4.429060,6.090664,-6.267686,0.0,5.920514,7.908811,0.0,9.372866,2015,0
1,1.296315,0.693147,4.041465,5.836494,5.367725,0.0,6.058968,7.899137,0.0,9.400984,2016,0
2,0.876094,0.693147,4.045652,5.850603,5.367725,0.0,6.172390,7.894630,0.0,9.384136,2017,0
3,0.788821,0.693147,4.314058,6.014482,5.367725,0.0,6.285673,7.907985,0.0,9.406473,2018,0
4,2.208824,0.693147,4.416512,6.197275,5.367725,0.0,6.384537,7.847541,0.0,9.430781,2019,0


In [6]:
train.shape

(3601, 13)

In [8]:
train_og.shape

(5603, 31)

## LSTM

In [19]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# Define target and feature columns
target_col = "Peningkatan Suhu Rata - Rata ° C"
feature_cols = ["Emisi Savanna Api", "Emisi Tanah Organik Yang Dikeringkan (Co2)", 
                "Emisi Pembuatan Pestisida", "Emisi Transportasi Makanan", "Lahan Hutan", 
                "Konversi Hutan Bersih", "Emisi Ritel Makanan", "Emisi Pupuk Kandang Di Padang Rumput", 
                "Emisi Kebakaran Di Tanah Organik", "Emisi Total"]

# Drop missing values
train.dropna(inplace=True)
test.dropna(inplace=True)

# Scale features
scaler = MinMaxScaler()
train[feature_cols] = scaler.fit_transform(train[feature_cols])
test[feature_cols] = scaler.transform(test[feature_cols])

# Function to create sequences
def create_sequences(data, target_col, seq_length, is_train=True):
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        x = data.iloc[i:(i + seq_length)][feature_cols].values
        xs.append(x)
        if is_train:
            ys.append(data.iloc[i + seq_length][target_col])  # Only for training
    if is_train:
        return np.array(xs), np.array(ys)
    return np.array(xs)  # Only return X if test data

# Set sequence length
seq_length = 10

# Create sequences for training
X_train, y_train = create_sequences(train, target_col, seq_length, is_train=True)

# Create sequences for testing (ONLY X_test, since y_test is missing)
X_test = create_sequences(test, target_col, seq_length, is_train=False)

# Reshape input for LSTM
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], len(feature_cols)))
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], len(feature_cols)))

# Build LSTM model
model = Sequential([
    LSTM(50, activation='relu', input_shape=(seq_length, len(feature_cols))),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')

# Train the model with batch size of 64
history = model.fit(X_train, y_train, epochs=200, batch_size=64, verbose=1)

# Make predictions
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Compute MAPE only for the training set
mape_train = np.mean(np.abs((y_train - y_pred_train.flatten()) / y_train)) * 100

c:\Users\HP\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



Epoch 1/200
57/57 ━━━━━━━━━━━━━━━━━━━━ 13s 27ms/step - loss: 0.4308
Epoch 2/200
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.2438
Epoch 3/200
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 0.2406
Epoch 4/200
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.2338
Epoch 5/200
57/57 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - loss: 0.2316
Epoch 6/200
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 0.2379
Epoch 7/200
57/57 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - loss: 0.2436
Epoch 8/200
57/57 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - loss: 0.2209
Epoch 9/200
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 0.2315
Epoch 10/200
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 0.2244
Epoch 11/200
57/57 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - loss: 0.2277
Epoch 12/200
57/57 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - loss: 0.2176
Epoch 13/200
57/57 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - loss: 0.2117
Epoch 14/200
57/57 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - loss: 0.2062
Epoch 15/200
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - l

In [20]:
print(f'MAPE on Training Set: {mape_train:.2f}%')

MAPE on Training Set: 95.21%
